# 🔍 Google Fact Check Tools API — Interactive Claim Validator

**Code Part 2 of the Social Media Analytics & Fact-Checking Toolkit**

This notebook provides a frontend interactive Google Colab application that leverages the **Google Fact Check Tools API** to audit user-submitted claims against verified global databases.

---

### 🚀 Quick Start
1. Get a free API key from [Google Cloud Console](https://console.cloud.google.com/) and enable the **Fact Check Tools API**
2. Run **Cell 1** to install/import dependencies
3. Run **Cell 2** to enter your API key securely (it will not be shown or stored)
4. Run **Cell 3** to launch the interactive checker widget

> ⚠️ **Security note:** Never hardcode your API key in notebook cells. Use `getpass` (as shown in Cell 2) to keep it out of version control.

In [ ]:
# Cell 1 — Install and import dependencies
# Uncomment the pip line below if running in Google Colab for the first time
# !pip install ipywidgets requests --quiet

import requests
import getpass
from ipywidgets import widgets
from IPython.display import display, HTML, clear_output

print('✅ Libraries loaded successfully.')

In [ ]:
# Cell 2 — Enter your Google Fact Check Tools API key (hidden input)
# Get your key at: https://console.cloud.google.com/
# Then enable: APIs & Services > Fact Check Tools API

API_KEY = getpass.getpass('🔑 Enter your Google Fact Check Tools API Key: ')
print('API key stored in memory. Proceed to Cell 3 to launch the widget.')

In [ ]:
# Cell 3 — Core API logic + Interactive Widget UI

def fetch_fact_checks(query, api_key, lang='en'):
    """Queries the Google Fact Check Tools API for claims matching the query."""
    url = 'https://factchecktools.googleapis.com/v1alpha1/claims:search'
    params = {
        'query': query,
        'key': api_key,
        'languageCode': lang
    }
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as http_err:
        return {'error': f'HTTP error: {http_err} — check your API key restrictions.'}
    except Exception as err:
        return {'error': f'Unexpected error: {err}'}


def display_results(data, query):
    """Formats the API response into clean, color-coded HTML verdict cards."""
    if 'error' in data:
        display(HTML(f"<p style='color:red;'><b>Error:</b> {data['error']}</p>"))
        return

    if 'claims' not in data or not data['claims']:
        display(HTML(
            f"<div style='padding:10px;background:#f8d7da;color:#721c24;border-radius:5px;'>"
            f"No fact-check records found for: <b>'{query}'</b>. "
            f"Try different keywords or a more specific phrasing.</div>"
        ))
        return

    html_output = f"<h3>Fact Check Results for: <i>'{query}'</i></h3>"

    for idx, claim_item in enumerate(data['claims'], 1):
        claim_text = claim_item.get('text', 'N/A')
        claimant   = claim_item.get('claimant', 'Unknown Source')
        claim_date = claim_item.get('claimDate', 'Unknown date')[:10]

        html_output += (
            f"<div style='border:1px solid #ddd;padding:15px;margin-bottom:15px;"
            f"border-radius:8px;background:#fafafa;font-family:sans-serif;'>"
            f"<p style='font-size:16px;margin:0 0 8px 0;'><b>[{idx}] Claim:</b> \"{claim_text}\"</p>"
            f"<p style='font-size:13px;color:#666;margin:0 0 12px 0;'>"
            f"<i>Stated by {claimant} on {claim_date}</i></p>"
        )

        for review in claim_item.get('claimReview', []):
            publisher  = review.get('publisher', {}).get('name', 'Unknown Publisher')
            rating     = review.get('textualRating', 'No Rating Provided').upper()
            review_url = review.get('url', '#')
            title      = review.get('title', 'View Review Article')

            # Colour-code the verdict badge
            badge_color = '#6c757d'  # grey (unknown)
            if any(w in rating for w in ['FALSE', 'FAKE', 'INCORRECT', 'MISLEADING']):
                badge_color = '#dc3545'  # red
            elif any(w in rating for w in ['TRUE', 'CORRECT', 'ACCURATE']):
                badge_color = '#28a745'  # green
            elif any(w in rating for w in ['PART', 'MIXTURE', 'MOSTLY']):
                badge_color = '#ffc107'  # amber

            html_output += (
                f"<div style='margin-left:20px;padding:10px;border-left:4px solid {badge_color};"
                f"background:#fff;margin-top:8px;'>"
                f"<p style='margin:0 0 5px 0;'><b>Reviewer:</b> {publisher}</p>"
                f"<p style='margin:0 0 5px 0;'><b>Verdict:</b> "
                f"<span style='background:{badge_color};color:white;padding:2px 6px;"
                f"border-radius:3px;font-size:12px;font-weight:bold;'>{rating}</span></p>"
                f"<p style='margin:0;'><a href='{review_url}' target='_blank' "
                f"style='color:#007bff;text-decoration:none;font-weight:500;'>"
                f"Link: {title}</a></p></div>"
            )

        html_output += '</div>'

    display(HTML(html_output))


# ── Widget UI ────────────────────────────────────────────────────────────────

input_text = widgets.Text(
    value='moon landing is fake',
    placeholder='Type a claim or keyword...',
    description='Claim:',
    layout=widgets.Layout(width='55%')
)

lang_dropdown = widgets.Dropdown(
    options=[
        ('English', 'en'),
        ('German', 'de'),
        ('French', 'fr'),
        ('Spanish', 'es'),
        ('Portuguese', 'pt'),
        ('Arabic', 'ar'),
    ],
    value='en',
    description='Language:',
    layout=widgets.Layout(width='18%')
)

button = widgets.Button(
    description='Check Claim',
    button_style='primary',
    icon='search'
)

output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output()
        if not input_text.value.strip():
            print('Please enter a claim or keyword to check.')
            return
        results = fetch_fact_checks(input_text.value, API_KEY, lang=lang_dropdown.value)
        display_results(results, input_text.value)

button.on_click(on_button_clicked)

display(HTML('<h2>Google Fact Check Tools API — Interactive Checker</h2>'))
display(widgets.HBox([input_text, lang_dropdown, button]))
display(output)